In [1]:
from sklearn.cluster import AgglomerativeClustering
from scipy import stats
!pip install scipy



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# ============================================================
#     ELSS FUND CLUSTERING PIPELINE (NUMERIC-SAFE VERSION)
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# ----------------- CONFIG -----------------
INPUT_CSV = "engineered_features_elss.csv"
GROUP_COL = "Scheme Code"
DATE_COL  = "Date"
NAV_COL   = "NAV"

OUT_DIR = "clustering_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

PCA_N_COMPONENTS = 3
K_MIN, K_MAX = 2, 8
RANDOM_STATE = 42
# -------------------------------------------

# 1) Load data
df = pd.read_csv(INPUT_CSV)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.sort_values([GROUP_COL, DATE_COL]).reset_index(drop=True)
print("Loaded:", df.shape)

# 2) Aggregate per scheme (select only columns we expect to exist)
agg_funcs = {
    NAV_COL: ["mean", "std", "min", "max"],
    "target_next_return": ["mean", "std"],
    "nav_diff_1": ["mean", "std"],
    "nav_roll_mean_7": ["mean"],
    "nav_roll_mean_30": ["mean"],
    "nav_roll_std_7": ["mean"],
    "time_index": ["max"],
    "time_index_norm": ["mean"],
    "time_index_sq": ["mean"],
    "t_days": ["max"],
    "nav_expanding_std": ["mean"],
    "scheme_nav_mean": ["first"],
    "scheme_nav_std": ["first"]
}
# keep only keys that exist in df
agg_funcs = {k: v for k, v in agg_funcs.items() if k in df.columns}

scheme_agg = df.groupby(GROUP_COL).agg(agg_funcs)
scheme_agg.columns = ["_".join(col).strip() for col in scheme_agg.columns.values]
scheme_agg = scheme_agg.reset_index()

print("Per-scheme aggregated:", scheme_agg.shape)

# 3) Add skew & kurtosis (pandas-safe implementation)
def get_skew_kurt(series):
    return pd.Series({
        "ret_skew": series.skew(),
        "ret_kurt": series.kurtosis()
    })

if "target_next_return" in df.columns:
    sk = df.groupby(GROUP_COL)["target_next_return"].apply(get_skew_kurt).reset_index()
    scheme_agg = scheme_agg.merge(sk, on=GROUP_COL, how="left")

# 4) Add recent momentum (last 5 returns) if available
if "target_next_return" in df.columns:
    def recent_mom(g, n=5):
        return g.sort_values(DATE_COL).tail(n)["target_next_return"].mean()
    recent = df.groupby(GROUP_COL).apply(recent_mom).rename("recent_ret_mean").reset_index()
    scheme_agg = scheme_agg.merge(recent, on=GROUP_COL, how="left")

# 5) Add Fourier means if available
four_cols = [c for c in df.columns if "fourier" in c or "month_sin" in c or "month_cos" in c]
if four_cols:
    fc = df.groupby(GROUP_COL)[four_cols].mean().reset_index()
    fc = fc.rename(columns={c: f"{c}_mean" for c in four_cols})
    scheme_agg = scheme_agg.merge(fc, on=GROUP_COL, how="left")

print("Final scheme table:", scheme_agg.shape)

# 6) Prepare feature matrix
id_col = GROUP_COL
features = [c for c in scheme_agg.columns if c != id_col]

X = scheme_agg[features].copy()

# ---- NUMERIC-SAFE handling ----
# separate numeric and non-numeric columns
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in X.columns if c not in numeric_cols]

# Fill numeric NAs with median (only on numeric columns)
if len(numeric_cols) > 0:
    med = X[numeric_cols].median()
    X[numeric_cols] = X[numeric_cols].fillna(med)

# For any remaining non-numeric columns (rare), fill with mode or drop them:
# We'll drop non-numeric columns here because cluster features should be numeric.
if len(non_numeric_cols) > 0:
    print("Dropping non-numeric columns from clustering features:", non_numeric_cols)
    X = X.drop(columns=non_numeric_cols)

# Remove columns with near-zero variance (numeric only now)
stds = X.std(axis=0, ddof=0)
keep_cols = stds[stds > 1e-8].index.tolist()
X = X[keep_cols]

print("Features after cleaning (numeric only):", X.shape)

# If after cleaning there are too few features, fail early
if X.shape[1] < 2:
    raise ValueError("Not enough numeric features to cluster after cleaning. Check input dataset or feature generation.")

# 7) Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, os.path.join(OUT_DIR, "scaler.joblib"))

# 8) PCA for visualization
pca = PCA(n_components=min(PCA_N_COMPONENTS, X_scaled.shape[1]), random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
joblib.dump(pca, os.path.join(OUT_DIR, "pca.joblib"))

print("PCA cumulative variance:", pca.explained_variance_ratio_.cumsum())

# 9) Silhouette scan to choose K (bounded by number of schemes)
n_schemes = X_scaled.shape[0]
K_MAX_SAFE = min(K_MAX, max(2, n_schemes - 1))
sil_scores = {}
for k in range(K_MIN, K_MAX_SAFE + 1):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X_scaled)
    # If too few samples for silhouette, skip
    try:
        sil = silhouette_score(X_scaled, labels)
    except Exception as e:
        sil = -1
    sil_scores[k] = sil
    print(f"K={k} => silhouette={sil:.4f}")

best_k = max(sil_scores, key=sil_scores.get)
print("Best K:", best_k)

# 10) Fit final KMeans
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20)
scheme_agg["cluster_kmeans"] = kmeans.fit_predict(X_scaled)
joblib.dump(kmeans, os.path.join(OUT_DIR, "kmeans_model.joblib"))

# 11) Hierarchical clustering (validation)
agg = AgglomerativeClustering(n_clusters=best_k)
scheme_agg["cluster_agg"] = agg.fit_predict(X_scaled)

# 12) Cluster profile (use numeric features)
profile = scheme_agg.groupby("cluster_kmeans")[X.columns.tolist()].mean()
profile.to_csv(os.path.join(OUT_DIR, "cluster_profile.csv"))
print("Cluster profile saved.")

# 13) Save assignments
scheme_agg[[GROUP_COL, "cluster_kmeans", "cluster_agg"]].to_csv(
    os.path.join(OUT_DIR, "scheme_clusters.csv"), index=False
)
print("Scheme assignments saved:", os.path.join(OUT_DIR, "scheme_clusters.csv"))

# 14) PCA scatter
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1],
                hue=scheme_agg["cluster_kmeans"].astype(str), palette="tab10", s=60)
plt.title("PCA Clusters (KMeans)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pca_clusters.png"))
plt.close()

print("Clustering Complete. All outputs stored in:", OUT_DIR)


Loaded: (319086, 55)
Per-scheme aggregated: (269, 19)
Final scheme table: (538, 28)
Dropping non-numeric columns from clustering features: ['level_1']
Features after cleaning (numeric only): (538, 26)
PCA cumulative variance: [0.45632188 0.67038466 0.74781795]
K=2 => silhouette=0.7411
K=3 => silhouette=0.5697
K=4 => silhouette=0.5608
K=5 => silhouette=0.4364
K=6 => silhouette=0.5686
K=7 => silhouette=0.4689
K=8 => silhouette=0.4808
Best K: 2
Cluster profile saved.
Scheme assignments saved: clustering_outputs\scheme_clusters.csv
Clustering Complete. All outputs stored in: clustering_outputs
